<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook A02: Visualizing Time Series</h2>
</div>

Worked solutions to the 5 exercises in
[Notebook A02: Visualizing Time Series](../notebooks/A02_Basic_plotting.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

df = pd.read_parquet(nb_config.CDC_TEMP_PATH)

bb_ser = df["Brandenburg/Berlin"]
bb_ser.name = "Brandenburg/Berlin"

print(f"{len(df)} months x {df.shape[1]} regions")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Plot the full Deutschland series alongside its year-on-year difference (`bb_ser.diff(12)`). What does the difference series tell you that the raw series does not?

In [ ]:
germany = df["Deutschland"]
year_on_year = germany.diff(12)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(germany, color="steelblue", linewidth=0.7)
axes[0].set_title("Deutschland: monthly temperature", fontsize=13, fontweight="bold")
axes[0].set_ylabel("°C")

axes[1].plot(year_on_year, color="seagreen", linewidth=0.7)
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].set_title("Change against the same month a year earlier", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("°C")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

print(f"raw       mean {germany.mean():6.2f}   std {germany.std():.2f}   "
      f"range {germany.min():.1f} to {germany.max():.1f}")
print(f"diff(12)  mean {year_on_year.mean():6.3f}   std {year_on_year.std():.2f}   "
      f"range {year_on_year.min():.1f} to {year_on_year.max():.1f}")

The raw series is dominated by one thing: the seasonal cycle, swinging roughly 20 °C every year. It is so
large that it hides everything else, and the eye cannot do much with it beyond confirming that summers are
warmer than winters.

The difference series removes that cycle by construction, because each month is compared with the same
month a year earlier. What is left is a series centred on zero with about a third of the spread, and it
answers a different question: **was this month warmer or colder than the same month last year?**

Two things become visible that the raw plot cannot show.

**Individual extreme years.** The largest swings, around ±13 °C, are single months that were far out of
line with the year before. February 1929 and January 1940 are the record cold anomalies; February 1957 is
the largest warm one. On the raw plot these are simply cold winters among many cold winters.

**The absence of a visible trend.** The difference series looks stationary, hovering around zero across
145 years. That is not evidence that there is no warming — a steady trend of about 0.5 °C per decade shows
up here as a mean of just **+0.02 °C**, which is invisible against a standard deviation of 2.5.
Differencing is a filter, and it removes slow signals along with the seasonality. Use it to see the fast
changes, and the rolling means from section 3 of the notebook to see the slow ones.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Apply a 12-month rolling mean to all 17 regions and plot them on a single chart (no confidence interval needed). Which regions show the steepest upward trend since 1980?

In [ ]:
smoothed = df.rolling(12).mean()

fig, ax = plt.subplots(figsize=(14, 5))

for region in df.columns:
    ax.plot(smoothed[region]["1980":], linewidth=0.9, alpha=0.75)

ax.set_title("12-month rolling mean, all 17 regions, 1980 onwards",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Seventeen lines on one chart is too many to read a ranking from, which is the honest reaction to that
plot. Fitting a slope to each one answers the question the picture cannot.

In [ ]:
def warming_per_decade(series, start="1980"):
    """Least-squares slope of the smoothed series, in °C per decade."""
    values = series.rolling(12).mean()[start:].dropna()
    months = np.arange(len(values))
    slope_per_month = np.polyfit(months, values.to_numpy(), 1)[0]
    return slope_per_month * 120


trends = pd.Series(
    {region: warming_per_decade(df[region]) for region in df.columns}
).sort_values(ascending=False)

print("Warming since 1980 (°C per decade):\n")
print(trends.round(3).to_string())

**Thüringen is steepest at 0.506 °C per decade**, with Mecklenburg-Vorpommern, Thüringen/Sachsen-Anhalt
and Bayern all within 0.02 of it. **Nordrhein-Westfalen is slowest at 0.425**, and it is the only region
that stands clearly apart from the rest.

The honest headline is how *little* the regions differ. The whole spread is 0.08 °C per decade across
seventeen regions, and fourteen of them sit inside a band of 0.05. This is one climate signal observed
seventeen times, not seventeen different signals, and the ranking should not be read as though the gaps
between neighbouring entries mean much.

It is worth resisting a tempting explanation here. A maritime-versus-inland story would predict that the
coastal regions warm slowest, moderated by the sea. **The data does not support it**:
Mecklenburg-Vorpommern sits on the Baltic and ranks second fastest, while Nordrhein-Westfalen is inland and
ranks last. There is a loose east-to-west gradient, with the eastern regions generally above the western
ones, but with a total spread this small and a crude straight-line fit, that is an observation rather than
a finding. If it mattered, the next step would be to ask whether the differences survive a proper
uncertainty estimate — and on this evidence most of them probably would not.

Two notes on the method. The slope is fitted to the *smoothed* series, since fitting raw monthly data
gives the same answer with far more variance. And a straight line is a crude summary of a curve that
steepens after about 1990: adequate for ranking, not for extrapolating.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Build the same box plot for the `Sachsen` and `Schleswig-Holstein` series side by side on two subplots. Which region has more variable winters?

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

regions = ["Sachsen", "Schleswig-Holstein"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

for ax, region in zip(axes, regions):
    series = df[region]
    sns.boxplot(
        x=pd.Categorical(series.index.month_name(), categories=month_order, ordered=True),
        y=series,
        ax=ax,
    )
    ax.set_title(region, fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Temperature (°C)" if region == regions[0] else "")
    ax.tick_params(axis="x", rotation=90)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
winter = df[df.index.month.isin([12, 1, 2])]
summer = df[df.index.month.isin([6, 7, 8])]

spread = pd.DataFrame({
    "winter std": winter[regions].std(),
    "winter IQR": winter[regions].quantile(0.75) - winter[regions].quantile(0.25),
    "summer std": summer[regions].std(),
})

spread.round(2)

**Sachsen has the more variable winters**, with a standard deviation of 2.82 °C against
Schleswig-Holstein's 2.58, and a wider interquartile range to match.

The reason is the same one that ordered the trends in the previous exercise. Schleswig-Holstein sits
between the North Sea and the Baltic, and water has an enormous heat capacity: it warms and cools slowly,
which keeps the air above it within a narrower range. Sachsen is inland, far from that moderation, and
therefore closer to a continental climate where winters can be genuinely cold or unseasonably mild.

Notice that the summer column shows much less difference between the two. Maritime moderation matters most
when the contrast between land and sea temperature is largest, which is winter. **A single "variability"
number for a region would have hidden that**, and it is the reason to compare distributions by month
rather than in aggregate.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-4">Exercise 4</h3>
</div>

> Build lag plots for lags 3, 6, 9, and 12. Does the relationship at lag 3 or lag 9 look linear or more curved? What does that tell you about the data?

In [ ]:
lags = [3, 6, 9, 12]

fig, axes = plt.subplots(1, len(lags), figsize=(15, 4), sharey=True)

for ax, lag in zip(axes, lags):
    ax.scatter(bb_ser, bb_ser.shift(lag), alpha=0.25, s=8, color="steelblue")
    correlation = bb_ser.corr(bb_ser.shift(lag))
    ax.set_title(f"Lag {lag}   r = {correlation:+.3f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value at t")
    if lag == lags[0]:
        ax.set_ylabel("Value at t + lag")
    ax.grid(linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

for lag in lags:
    pearson = bb_ser.corr(bb_ser.shift(lag))
    spearman = bb_ser.corr(bb_ser.shift(lag), method="spearman")
    print(f"lag {lag:2d}:  pearson {pearson:+.3f}   spearman {spearman:+.3f}")

**Neither.** At lags 3 and 9 the relationship is not linear and it is not curved — there is no functional
relationship at all. The points form a **ring**, and the correlation is essentially zero: +0.009 at lag 3
and +0.007 at lag 9.

That is a more interesting answer than the question implies, and the ring is the reason. Three months is a
quarter of the annual cycle, so a cold January maps to a mild April while a warm July maps to a mild
October. Mild values at *t* + 3 therefore arise from both extremes at *t*, and every value on the
horizontal axis is paired with a value that could be anywhere in the middle of the vertical one. Trace the
year around and the points close into a loop.

The reported correlations make the point sharply. Pearson is +0.009, and **Spearman is +0.010** — almost
identical. Spearman would detect a relationship that was monotone but curved, so the fact that it also
reads zero rules that out. There is genuine structure here, and it is perfectly deterministic given the
seasonal cycle, yet no correlation coefficient can see it.

The lesson generalises well past this plot. **A correlation of zero means no linear association, not no
relationship.** Lags 6 and 12 give -0.911 and +0.926, so the ACF of this series will show large
alternating spikes and nothing at all at the quarter-cycle lags — which is exactly what section 7 of the
notebook shows.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-5">Exercise 5</h3>
</div>

> Compute and plot the ACF for the year-on-year differenced series (`bb_ser.diff(12).dropna()`). How does it compare to the ACF of the raw series? What has differencing removed?

In [ ]:
differenced = bb_ser.diff(12).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

plot_acf(bb_ser.dropna(), lags=36, ax=axes[0], color="steelblue",
         vlines_kwargs={"colors": "steelblue"})
axes[0].set_title("ACF: raw series", fontsize=13, fontweight="bold")

plot_acf(differenced, lags=36, ax=axes[1], color="seagreen",
         vlines_kwargs={"colors": "seagreen"})
axes[1].set_title("ACF: differenced at lag 12", fontsize=13, fontweight="bold")

for ax in axes:
    ax.set_xlabel("Lag (months)")
    ax.set_ylim(-1.1, 1.1)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import acf

raw_acf = acf(bb_ser.dropna(), nlags=26)
differenced_acf = acf(differenced, nlags=26)

pd.DataFrame(
    {"raw": raw_acf[[1, 3, 6, 12, 18, 24]], "differenced": differenced_acf[[1, 3, 6, 12, 18, 24]]},
    index=[1, 3, 6, 12, 18, 24],
).rename_axis("lag").round(3)

The two plots could hardly look less alike.

**The raw ACF oscillates without decaying.** +0.92 at lag 12, -0.91 at lag 6, +0.92 at 24, -0.90 at 18,
and so on for as many lags as you care to compute. That sustained alternation is the signature of a strong,
stable seasonal cycle: the series is almost perfectly correlated with itself a year ago and almost
perfectly anti-correlated half a year ago. Note also that it never dies away, which is the ACF telling you
the series is not stationary.

**The differenced ACF is nearly empty.** Every one of those spikes is gone. What remains is +0.21 at lag 1,
and a single **negative** spike of -0.50 at lag 12.

So differencing removed the seasonality, which is what it was for. The interesting part is what it left
behind, and neither piece is a leftover of the original signal:

- The **-0.50 at lag 12** is created by the differencing itself. Subtracting the value twelve months
  earlier makes consecutive differenced values share a term with opposite signs, which induces exactly
  this negative correlation at the seasonal lag. Notebook
  [B02](../notebooks/B02_ARIMA_models.ipynb) reads that spike as the signature of a seasonal MA(1) term,
  and fits one.
- The **+0.21 at lag 1** is real short-range persistence: a month warmer than its counterpart last year
  tends to be followed by another one. That is what an AR(1) term captures, and B02 fits that too.

Read together, the two plots are the whole argument for differencing before modelling. The raw series has
correlation structure so dominated by the season that nothing else is visible; the differenced one has a
short, readable structure that a small model can fit.

---

Back to [Notebook A02](../notebooks/A02_Basic_plotting.ipynb), or on to
[Notebook A03](../notebooks/A03_Handling_missing_data.ipynb).